# MSG-SE-DenseNet-EfficentTemp-GAN：三种时间降尺度策略 mid 时次指标对比

本 notebook 用于对比同一个最优模型 **MSG-SE-DenseNet-EfficentTemp-GAN** 的三种时间降尺度策略：

1. **Direct**：直接法，即不带后缀的 `100km_1day_to_50km_6hour` 模型；
2. **Separate**：分别法，即 `time6`、`time12`、`time18` 三个模型分别预测中间时次；
3. **Indirect**：间接法，即先用 `100km_1day_to_50km_12hour` 得到 50 km / 12 h 结果，再输入 `50km_12hour_to_50km_6hour` 得到 50 km / 6 h 结果。

只计算 **mid 时次**：time6、time12、time18。  
输出包括：

- 三种策略的总体指标（5 个变量、3 个 mid 时次平均）；
- 三种策略在每个变量上的指标（每个变量的 time6/time12/time18 平均）；
- 每个变量每个 mid 时次的详细指标。


In [1]:
# ============================================================
# 公共代码框 1：
# 读取 5 变量 + 构造多变量 HR/LR + 标准化 + 测试集 memmap
# ============================================================
# 输入 testx : (N, 58, 94, 10)
# 输出 testy : (N, 116, 188, 25)
#
# 变量顺序：
# VAR_NAMES = ["slp", "z300", "z500", "u10", "v10"]
#
# testx 通道：
# slp_t, slp_t4, z300_t, z300_t4, z500_t, z500_t4, u10_t, u10_t4, v10_t, v10_t4
#
# testy 通道：
# slp_t..slp_t4, z300_t..z300_t4, z500_t..z500_t4, u10_t..u10_t4, v10_t..v10_t4
# ============================================================

import os
import gc
import json
import warnings
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import xarray as xr
from tqdm import tqdm
from numpy.lib.format import open_memmap

warnings.filterwarnings("ignore")

# ============================================================
# 1. 路径与基本参数
# ============================================================
DATA_DIR = Path(r"H:\ERA5-6hour")
MODEL_DIR = Path(r"E:\Dr_Research\model")
RESULT_DIR = Path(r"E:\Dr_Research\result")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

TMP_DIR = RESULT_DIR / "_tmp_downscaling_5vars_data"
TMP_DIR.mkdir(parents=True, exist_ok=True)

TEST_SIZE = 0.2

TIME_START = "1980-01-01T00:00:00"
TIME_END   = "2014-12-31T18:00:00"

LAT_RANGE = (-5.0, 53.0)
LON_RANGE = (93.0, 187.0)

VAR_NAMES = ["slp", "z300", "z500", "u10", "v10"]
FEATURE_HOURS = np.array([0, 6, 12, 18, 24], dtype=np.int32)

ALL_FEATURE_INDEX = np.arange(25, dtype=np.int32)
SPATIAL_ONLY_FEATURE_INDEX = np.array(
    [0, 4, 5, 9, 10, 14, 15, 19, 20, 24],
    dtype=np.int32
)
MID_FEATURE_INDEX = np.array(
    [i for i in range(25) if i not in set(SPATIAL_ONLY_FEATURE_INDEX.tolist())],
    dtype=np.int32
)

TESTX_NPY = TMP_DIR / "testx_data_standardized.npy"
TESTY_NPY = TMP_DIR / "testy_data_standardized.npy"
META_NPZ  = TMP_DIR / "metadata_data_standardized.npz"

# 重新构造数据时设为 True；已经构造过、只想重新算模型或指标时，可改为 False
REBUILD_DATA = False

VARIABLE_CONFIGS = {
    "slp": {
        "file": "Mean-sea-level-pressure-1980-2024.nc",
        "nc_var_candidates": ["msl", "psl", "slp", "mean_sea_level_pressure"],
        "level": None,
    },
    "z300": {
        "file": "Geopotential-300hpa-1980-2024.nc",
        "nc_var_candidates": ["z", "z300", "g300", "geopotential"],
        "level": 300,
    },
    "z500": {
        "file": "Geopotential-500hpa-1980-2024.nc",
        "nc_var_candidates": ["z", "z500", "g500", "geopotential"],
        "level": 500,
    },
    "u10": {
        "file": "10m-u-component-of-wind-1980-2024.nc",
        "nc_var_candidates": ["u10", "u", "10u", "u_component_of_wind_10m"],
        "level": None,
    },
    "v10": {
        "file": "10m-v-component-of-wind-1980-2024.nc",
        "nc_var_candidates": ["v10", "v", "10v", "v_component_of_wind_10m"],
        "level": None,
    },
}


# ============================================================
# 2. 读取与预处理函数
# ============================================================
def _find_dim_name(da, dim_keywords):
    for d in da.dims:
        dl = d.lower()
        if any(k in dl for k in dim_keywords):
            return d
    raise ValueError(f"无法在 dims={da.dims} 中找到维度关键词: {dim_keywords}")


def _guess_var_name(ds, candidates):
    for v in candidates:
        if v in ds.data_vars:
            return v

    valid_vars = []
    for v in ds.data_vars:
        if ds[v].ndim >= 3:
            valid_vars.append(v)

    if len(valid_vars) == 1:
        return valid_vars[0]

    raise ValueError(
        f"无法自动识别变量名。候选={candidates}, 文件内变量={list(ds.data_vars)}"
    )


def _select_level_if_needed(da, level_value):
    if level_value is None:
        return da

    level_dim_candidates = [
        d for d in da.dims
        if d.lower() in ["level", "pressure_level", "isobaricinhpa", "plev"]
        or "level" in d.lower()
        or "pressure" in d.lower()
    ]

    if len(level_dim_candidates) == 0:
        return da

    level_dim = level_dim_candidates[0]
    coord_values = da[level_dim].values

    if len(coord_values) == 1:
        return da.isel({level_dim: 0})

    return da.sel({level_dim: level_value}, method="nearest")


def _select_lat_lon(da, lat_range, lon_range):
    lat_dim = _find_dim_name(da, ["lat", "latitude"])
    lon_dim = _find_dim_name(da, ["lon", "longitude"])

    lat_values = da[lat_dim].values
    lon_values = da[lon_dim].values

    lat_min, lat_max = lat_range
    lon_min, lon_max = lon_range

    if lat_values[0] < lat_values[-1]:
        da = da.sel({lat_dim: slice(lat_min, lat_max)})
    else:
        da = da.sel({lat_dim: slice(lat_max, lat_min)})

    lon_values = da[lon_dim].values

    if lon_values.min() >= 0 and lon_values.max() > 180:
        da = da.sel({lon_dim: slice(lon_min, lon_max)})
    else:
        lon_min_180 = ((lon_min + 180) % 360) - 180
        lon_max_180 = ((lon_max + 180) % 360) - 180

        if lon_min_180 <= lon_max_180:
            da = da.sel({lon_dim: slice(lon_min_180, lon_max_180)})
        else:
            part1 = da.sel({lon_dim: slice(lon_min_180, 180)})
            part2 = da.sel({lon_dim: slice(-180, lon_max_180)})
            da = xr.concat([part1, part2], dim=lon_dim)

    return da


def _select_time_range(da, time_start, time_end):
    time_dim = _find_dim_name(da, ["time"])
    da = da.sortby(time_dim)

    try:
        da = da.sel({time_dim: slice(np.datetime64(time_start), np.datetime64(time_end))})
    except Exception:
        times = pd.to_datetime(da[time_dim].values)
        mask = (times >= pd.Timestamp(time_start)) & (times <= pd.Timestamp(time_end))
        da = da.isel({time_dim: np.where(mask)[0]})

    return da


def read_one_variable(cfg):
    nc_path = DATA_DIR / cfg["file"]
    if not nc_path.exists():
        raise FileNotFoundError(f"文件不存在: {nc_path}")

    ds = xr.open_dataset(nc_path)

    var_name = _guess_var_name(ds, cfg["nc_var_candidates"])
    da = ds[var_name]

    da = _select_level_if_needed(da, cfg["level"])
    da = _select_lat_lon(da, LAT_RANGE, LON_RANGE)
    da = _select_time_range(da, TIME_START, TIME_END)

    time_dim = _find_dim_name(da, ["time"])
    lat_dim = _find_dim_name(da, ["lat", "latitude"])
    lon_dim = _find_dim_name(da, ["lon", "longitude"])

    da = da.transpose(time_dim, lat_dim, lon_dim)

    times = pd.to_datetime(da[time_dim].values)
    lat = da[lat_dim].values
    lon = da[lon_dim].values

    arr = da.values.astype(np.float32)

    ds.close()
    del ds, da
    gc.collect()

    return arr, times, lat, lon, var_name


def build_hr_lr_samples_single_var(arr, times, lat, lon):
    """
    从原始 0.25° / 6-hour 数据构造：
    HR: 0.5° / 6-hour, 5 个时次
    LR: 1.0° / 1-day, 2 个时次

    复现旧 notebook：
        arr_05 = arr[:, ::2, ::2]
        HR = arr_05[:, :-1, :-1]
        LR = arr_05[:, :-1:2, :-1:2]
    """

    T = arr.shape[0]

    arr_05 = arr[:, ::2, ::2]
    lat_05 = lat[::2]
    lon_05 = lon[::2]

    arr_hr_base = arr_05[:, :-1, :-1]
    arr_lr_base = arr_05[:, :-1:2, :-1:2]

    lat_hr = lat_05[:-1]
    lon_hr = lon_05[:-1]
    lat_lr = lat_05[:-1:2]
    lon_lr = lon_05[:-1:2]

    candidate_start = np.arange(0, T - 4)
    n = len(candidate_start)

    hr = np.empty(
        (n, arr_hr_base.shape[1], arr_hr_base.shape[2], 5),
        dtype=np.float32
    )
    lr = np.empty(
        (n, arr_lr_base.shape[1], arr_lr_base.shape[2], 2),
        dtype=np.float32
    )

    hr[..., 0] = arr_hr_base[candidate_start,     :, :]
    hr[..., 1] = arr_hr_base[candidate_start + 1, :, :]
    hr[..., 2] = arr_hr_base[candidate_start + 2, :, :]
    hr[..., 3] = arr_hr_base[candidate_start + 3, :, :]
    hr[..., 4] = arr_hr_base[candidate_start + 4, :, :]

    lr[..., 0] = arr_lr_base[candidate_start,     :, :]
    lr[..., 1] = arr_lr_base[candidate_start + 4, :, :]

    sample_times = times[candidate_start]

    return hr, lr, sample_times, lat_hr, lon_hr, lat_lr, lon_lr


def standardize_like_old_notebook(hr, lr):
    hr_mean = np.nanmean(hr, axis=0).astype(np.float32)
    hr_std  = np.nanstd(hr, axis=0).astype(np.float32)

    lr_mean = np.nanmean(lr, axis=0).astype(np.float32)
    lr_std  = np.nanstd(lr, axis=0).astype(np.float32)

    hr_std = np.where((hr_std == 0) | np.isnan(hr_std), 1.0, hr_std).astype(np.float32)
    lr_std = np.where((lr_std == 0) | np.isnan(lr_std), 1.0, lr_std).astype(np.float32)

    hr_norm = ((hr - hr_mean) / hr_std).astype(np.float32)
    lr_norm = ((lr - lr_mean) / lr_std).astype(np.float32)

    hr_range_crop = np.array(
        [
            np.nanmax(hr_norm[:, 1:-1, 1:-1, k]) - np.nanmin(hr_norm[:, 1:-1, 1:-1, k])
            for k in range(5)
        ],
        dtype=np.float32
    )
    hr_range_crop = np.where(
        (hr_range_crop == 0) | np.isnan(hr_range_crop),
        1.0,
        hr_range_crop
    ).astype(np.float32)

    return hr_norm, lr_norm, hr_mean, hr_std, lr_mean, lr_std, hr_range_crop


# ============================================================
# 3. 构造多变量 testx/testy
# ============================================================
def build_or_load_multivar_test_data():
    if (not REBUILD_DATA) and TESTX_NPY.exists() and TESTY_NPY.exists() and META_NPZ.exists():
        print("读取已有 memmap 数据。")
        testx = np.load(TESTX_NPY, mmap_mode="r")
        testy = np.load(TESTY_NPY, mmap_mode="r")
        meta = np.load(META_NPZ, allow_pickle=True)

        metadata = {
            "test_times": meta["test_times"],
            "lat": meta["lat"],
            "lon": meta["lon"],
            "lat_lr": meta["lat_lr"],
            "lon_lr": meta["lon_lr"],
            "hr_range_crop": meta["hr_range_crop"],
            "split_index": int(meta["split_index"]),
        }

        print("testx:", testx.shape)
        print("testy:", testy.shape)
        return testx, testy, metadata

    for p in [TESTX_NPY, TESTY_NPY, META_NPZ]:
        if p.exists():
            p.unlink()

    testx_mm = None
    testy_mm = None

    hr_range_crop_all = np.zeros((25,), dtype=np.float32)
    test_times_final = None
    lat_hr_final = None
    lon_hr_final = None
    lat_lr_final = None
    lon_lr_final = None
    split_index_final = None

    for vi, var_name in enumerate(VAR_NAMES):
        cfg = VARIABLE_CONFIGS[var_name]
        print(f"\n========== 读取并处理变量: {var_name} ==========")

        raw, times, lat, lon, nc_var_name = read_one_variable(cfg)

        print(f"[{var_name}] nc变量名:", nc_var_name)
        print(f"[{var_name}] raw shape:", raw.shape)
        print(f"[{var_name}] time:", times[0], "->", times[-1])
        print(f"[{var_name}] lat size:", len(lat), "lon size:", len(lon))

        hr, lr, sample_times, lat_hr, lon_hr, lat_lr, lon_lr = build_hr_lr_samples_single_var(
            raw, times, lat, lon
        )

        del raw
        gc.collect()

        print(f"[{var_name}] HR before norm:", hr.shape)
        print(f"[{var_name}] LR before norm:", lr.shape)

        hr_norm, lr_norm, hr_mean, hr_std, lr_mean, lr_std, hr_range_crop = standardize_like_old_notebook(hr, lr)

        split_index = int((1.0 - TEST_SIZE) * hr_norm.shape[0])

        if testx_mm is None:
            n_test = hr_norm.shape[0] - split_index
            h_lr, w_lr = lr_norm.shape[1], lr_norm.shape[2]
            h_hr, w_hr = hr_norm.shape[1], hr_norm.shape[2]

            testx_mm = open_memmap(
                TESTX_NPY,
                mode="w+",
                dtype=np.float32,
                shape=(n_test, h_lr, w_lr, 10)
            )
            testy_mm = open_memmap(
                TESTY_NPY,
                mode="w+",
                dtype=np.float32,
                shape=(n_test, h_hr, w_hr, 25)
            )

            test_times_final = np.array(sample_times[split_index:], dtype="datetime64[ns]")
            lat_hr_final = lat_hr.astype(np.float32)
            lon_hr_final = lon_hr.astype(np.float32)
            lat_lr_final = lat_lr.astype(np.float32)
            lon_lr_final = lon_lr.astype(np.float32)
            split_index_final = split_index

        else:
            if split_index != split_index_final:
                raise ValueError(f"{var_name} 的 split_index 与前面变量不一致。")
            if not np.array_equal(np.array(sample_times[split_index:], dtype="datetime64[ns]"), test_times_final):
                raise ValueError(f"{var_name} 的测试集时间与前面变量不一致。")

        x0 = vi * 2
        y0 = vi * 5

        testx_mm[..., x0:x0+2] = lr_norm[split_index:].astype(np.float32)
        testy_mm[..., y0:y0+5] = hr_norm[split_index:].astype(np.float32)
        hr_range_crop_all[y0:y0+5] = hr_range_crop

        testx_mm.flush()
        testy_mm.flush()

        print(f"[{var_name}] 写入 testx 通道 {x0}:{x0+2}")
        print(f"[{var_name}] 写入 testy 通道 {y0}:{y0+5}")

        del hr, lr, hr_norm, lr_norm, hr_mean, hr_std, lr_mean, lr_std
        gc.collect()

    np.savez(
        META_NPZ,
        test_times=test_times_final,
        lat=lat_hr_final,
        lon=lon_hr_final,
        lat_lr=lat_lr_final,
        lon_lr=lon_lr_final,
        hr_range_crop=hr_range_crop_all,
        split_index=np.array(split_index_final, dtype=np.int64),
    )

    testx_mm.flush()
    testy_mm.flush()

    testx = np.load(TESTX_NPY, mmap_mode="r")
    testy = np.load(TESTY_NPY, mmap_mode="r")
    meta = np.load(META_NPZ, allow_pickle=True)

    metadata = {
        "test_times": meta["test_times"],
        "lat": meta["lat"],
        "lon": meta["lon"],
        "lat_lr": meta["lat_lr"],
        "lon_lr": meta["lon_lr"],
        "hr_range_crop": meta["hr_range_crop"],
        "split_index": int(meta["split_index"]),
    }

    print("\n========== 多变量数据构造完成 ==========")
    print("testx:", testx.shape)
    print("testy:", testy.shape)
    print("test time:", metadata["test_times"][0], "->", metadata["test_times"][-1])
    print("HR lat/lon:", len(metadata["lat"]), len(metadata["lon"]))
    print("LR lat/lon:", len(metadata["lat_lr"]), len(metadata["lon_lr"]))

    return testx, testy, metadata


testx, testy, metadata = build_or_load_multivar_test_data()

print("\nALL_FEATURE_INDEX:", ALL_FEATURE_INDEX.tolist())
print("SPATIAL_ONLY_FEATURE_INDEX:", SPATIAL_ONLY_FEATURE_INDEX.tolist())
print("MID_FEATURE_INDEX:", MID_FEATURE_INDEX.tolist())


# ============================================================
# 4. 指标计算函数：R, MSE, SSIM, PSNR
# ============================================================
def feature_to_var_and_hour(feature_index):
    vi = int(feature_index // 5)
    hi = int(feature_index % 5)
    return VAR_NAMES[vi], int(FEATURE_HOURS[hi])


def calc_feature_r_mse_spatial_chunk(y_true, y_pred, feature_index, lat_chunk=12):
    """
    y_true/y_pred shape: (N, H, W, 25)
    对单个 feature 计算：
    1. 每个格点沿 time 计算 Pearson R，然后空间平均
    2. 每个格点沿 time 计算 MSE，然后空间平均
    """

    n, h, w, _ = y_true.shape

    r_sum = 0.0
    r_count = 0
    mse_sum = 0.0
    mse_count = 0

    for i0 in range(1, h - 1, lat_chunk):
        i1 = min(h - 1, i0 + lat_chunk)

        yt = np.asarray(y_true[:, i0:i1, 1:-1, feature_index], dtype=np.float64)
        yp = np.asarray(y_pred[:, i0:i1, 1:-1, feature_index], dtype=np.float64)

        mse_map = np.nanmean((yp - yt) ** 2, axis=0)
        valid_mse = np.isfinite(mse_map)
        if np.any(valid_mse):
            mse_sum += float(np.nansum(mse_map[valid_mse]))
            mse_count += int(np.sum(valid_mse))

        yt_mean = np.nanmean(yt, axis=0, keepdims=True)
        yp_mean = np.nanmean(yp, axis=0, keepdims=True)

        yt_anom = yt - yt_mean
        yp_anom = yp - yp_mean

        numerator = np.nansum(yt_anom * yp_anom, axis=0)
        denominator = np.sqrt(
            np.nansum(yt_anom ** 2, axis=0) *
            np.nansum(yp_anom ** 2, axis=0)
        )

        r_map = numerator / denominator
        valid_r = np.isfinite(r_map)
        if np.any(valid_r):
            r_sum += float(np.nansum(r_map[valid_r]))
            r_count += int(np.sum(valid_r))

        del yt, yp, yt_mean, yp_mean, yt_anom, yp_anom, numerator, denominator, r_map, mse_map
        gc.collect()

    r = r_sum / r_count if r_count > 0 else np.nan
    mse = mse_sum / mse_count if mse_count > 0 else np.nan

    return float(r), float(mse)


def calc_feature_ssim_psnr_tf(y_true, y_pred, feature_index, max_val, batch_size=16):
    """
    使用 TensorFlow 分 batch 计算单 feature 的 SSIM/PSNR。
    默认 batch_size 很小，避免显存或内存峰值过大。
    """

    import tensorflow as tf

    n = y_true.shape[0]

    if (not np.isfinite(max_val)) or max_val <= 0:
        max_val = 1.0

    ssim_sum = 0.0
    psnr_sum = 0.0
    count = 0

    with tf.device("/CPU:0"):
        for b0 in range(0, n, batch_size):
            b1 = min(n, b0 + batch_size)

            yt = np.asarray(y_true[b0:b1, 1:-1, 1:-1, feature_index], dtype=np.float32)
            yp = np.asarray(y_pred[b0:b1, 1:-1, 1:-1, feature_index], dtype=np.float32)

            yt = np.nan_to_num(yt, nan=0.0, posinf=0.0, neginf=0.0)[..., np.newaxis]
            yp = np.nan_to_num(yp, nan=0.0, posinf=0.0, neginf=0.0)[..., np.newaxis]

            ssim_batch = tf.image.ssim(yp, yt, max_val=float(max_val)).numpy()
            psnr_batch = tf.image.psnr(yp, yt, max_val=float(max_val)).numpy()

            valid_ssim = np.isfinite(ssim_batch)
            valid_psnr = np.isfinite(psnr_batch)

            if np.any(valid_ssim):
                ssim_sum += float(np.nansum(ssim_batch[valid_ssim]))
            if np.any(valid_psnr):
                psnr_sum += float(np.nansum(psnr_batch[valid_psnr]))

            count += int(b1 - b0)

            del yt, yp, ssim_batch, psnr_batch
            gc.collect()

    ssim = ssim_sum / count if count > 0 else np.nan
    psnr = psnr_sum / count if count > 0 else np.nan

    return float(ssim), float(psnr)


def calc_feature_metrics_table(y_true, y_pred, hr_range_crop, model_name, ssim_batch_size=16):
    rows = []

    for k in tqdm(range(25), desc=f"Feature metrics: {model_name}"):
        var_name, hour = feature_to_var_and_hour(k)

        r, mse = calc_feature_r_mse_spatial_chunk(
            y_true=y_true,
            y_pred=y_pred,
            feature_index=k,
            lat_chunk=12
        )

        ssim, psnr = calc_feature_ssim_psnr_tf(
            y_true=y_true,
            y_pred=y_pred,
            feature_index=k,
            max_val=float(hr_range_crop[k]),
            batch_size=ssim_batch_size
        )

        rows.append({
            "Model": model_name,
            "Variable": var_name,
            "Feature_Index": k,
            "Hour": hour,
            "R": r,
            "MSE": mse,
            "SSIM": ssim,
            "PSNR": psnr,
        })

        gc.collect()

    return pd.DataFrame(rows)


def aggregate_metrics_from_feature_table(feature_table, model_name):
    metric_cols = ["R", "MSE", "SSIM", "PSNR"]

    rows_all = []
    rows_mid = []

    for var_name in VAR_NAMES:
        all_part = feature_table[feature_table["Variable"] == var_name]
        mid_part = all_part[~all_part["Feature_Index"].isin(SPATIAL_ONLY_FEATURE_INDEX.tolist())]

        rows_all.append({
            "Model": model_name,
            "Variable": var_name,
            **all_part[metric_cols].mean(numeric_only=True).to_dict()
        })

        rows_mid.append({
            "Model": model_name,
            "Variable": var_name,
            **mid_part[metric_cols].mean(numeric_only=True).to_dict()
        })

    table_all_by_variable = pd.DataFrame(rows_all)
    table_mid_by_variable = pd.DataFrame(rows_mid)

    table_all_overall = (
        table_all_by_variable
        .groupby("Model", as_index=False)[metric_cols]
        .mean()
    )

    table_mid_overall = (
        table_mid_by_variable
        .groupby("Model", as_index=False)[metric_cols]
        .mean()
    )

    return table_all_overall, table_all_by_variable, table_mid_overall, table_mid_by_variable


def save_metrics_tables(feature_tables, out_xlsx):
    metric_cols = ["R", "MSE", "SSIM", "PSNR"]

    all_by_variable_list = []
    mid_by_variable_list = []

    for model_name, feature_table in feature_tables.items():
        _, all_by_variable, _, mid_by_variable = aggregate_metrics_from_feature_table(
            feature_table=feature_table,
            model_name=model_name
        )
        all_by_variable_list.append(all_by_variable)
        mid_by_variable_list.append(mid_by_variable)

    table2_all_by_variable = pd.concat(all_by_variable_list, axis=0, ignore_index=True)
    table4_mid_by_variable = pd.concat(mid_by_variable_list, axis=0, ignore_index=True)

    table1_all_overall = (
        table2_all_by_variable
        .groupby("Model", as_index=False)[metric_cols]
        .mean()
    )

    table3_mid_overall = (
        table4_mid_by_variable
        .groupby("Model", as_index=False)[metric_cols]
        .mean()
    )

    out_xlsx = Path(out_xlsx)
    out_xlsx.parent.mkdir(parents=True, exist_ok=True)

    with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
        table1_all_overall.to_excel(writer, sheet_name="table1_all_overall", index=False)
        table2_all_by_variable.to_excel(writer, sheet_name="table2_all_by_variable", index=False)
        table3_mid_overall.to_excel(writer, sheet_name="table3_mid_overall", index=False)
        table4_mid_by_variable.to_excel(writer, sheet_name="table4_mid_by_variable", index=False)

    prefix = out_xlsx.with_suffix("")
    table1_all_overall.to_csv(str(prefix) + "_table1_all_overall.csv", index=False, encoding="utf-8-sig")
    table2_all_by_variable.to_csv(str(prefix) + "_table2_all_by_variable.csv", index=False, encoding="utf-8-sig")
    table3_mid_overall.to_csv(str(prefix) + "_table3_mid_overall.csv", index=False, encoding="utf-8-sig")
    table4_mid_by_variable.to_csv(str(prefix) + "_table4_mid_by_variable.csv", index=False, encoding="utf-8-sig")

    print(f"\n指标表已保存: {out_xlsx}")

    print("\nTable 1: 全部时次总体指标，5变量平均")
    try:
        display(table1_all_overall)
    except Exception:
        print(table1_all_overall)

    print("\nTable 2: 全部时次分变量指标")
    try:
        display(table2_all_by_variable)
    except Exception:
        print(table2_all_by_variable)

    print("\nTable 3: 中间时次总体指标，不含 feature index = 0,4,5,9,10,14,15,19,20,24")
    try:
        display(table3_mid_overall)
    except Exception:
        print(table3_mid_overall)

    print("\nTable 4: 中间时次分变量指标")
    try:
        display(table4_mid_by_variable)
    except Exception:
        print(table4_mid_by_variable)

    return table1_all_overall, table2_all_by_variable, table3_mid_overall, table4_mid_by_variable


# ============================================================
# 5. 保存 nc 函数：每个变量保存为 data_slp/data_z300/.../data_v10
# ============================================================
def save_multivar_to_nc(arr, times, lat, lon, out_path, description, time_chunk=64, complevel=4):
    """
    arr shape: (N, H, W, 25)
    保存为：
        data_slp(time, feature, latitude, longitude)
        data_z300(time, feature, latitude, longitude)
        data_z500(time, feature, latitude, longitude)
        data_u10(time, feature, latitude, longitude)
        data_v10(time, feature, latitude, longitude)
    """

    from netCDF4 import Dataset, date2num

    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    n, h, w, f = arr.shape
    assert f == 25

    pd_times = pd.to_datetime(times)
    py_times = [t.to_pydatetime() for t in pd_times]

    time_units = f"hours since {py_times[0].strftime('%Y-%m-%d %H:%M:%S')}"
    calendar = "standard"
    time_values = date2num(py_times, units=time_units, calendar=calendar)

    ds = Dataset(out_path, "w", format="NETCDF4")

    ds.createDimension("time", None)
    ds.createDimension("feature", 5)
    ds.createDimension("latitude", h)
    ds.createDimension("longitude", w)

    tvar = ds.createVariable("time", "f8", ("time",))
    fvar = ds.createVariable("feature", "i4", ("feature",))
    hvar = ds.createVariable("feature_hour", "i4", ("feature",))
    latvar = ds.createVariable("latitude", "f4", ("latitude",))
    lonvar = ds.createVariable("longitude", "f4", ("longitude",))

    tvar[:] = time_values
    tvar.units = time_units
    tvar.calendar = calendar

    fvar[:] = np.arange(5, dtype=np.int32)
    hvar[:] = FEATURE_HOURS
    latvar[:] = lat.astype(np.float32)
    lonvar[:] = lon.astype(np.float32)

    ds.description = description
    ds.variable_order = ",".join(VAR_NAMES)
    ds.feature_description = "feature 0-4 corresponds to +0,+6,+12,+18,+24 hours for each variable"
    ds.spatial_resolution = "0.5 degree"
    ds.data_status = "standardized"

    encoding_kwargs = {
        "zlib": True,
        "complevel": complevel,
        "chunksizes": (min(time_chunk, n), 5, h, w),
    }

    for vi, var_name in enumerate(VAR_NAMES):
        y0 = vi * 5
        nc_var_name = f"data_{var_name}"

        vout = ds.createVariable(
            nc_var_name,
            "f4",
            ("time", "feature", "latitude", "longitude"),
            **encoding_kwargs
        )

        for t0 in tqdm(range(0, n, time_chunk), desc=f"保存 {nc_var_name}"):
            t1 = min(n, t0 + time_chunk)
            block = np.asarray(arr[t0:t1, :, :, y0:y0+5], dtype=np.float32).transpose(0, 3, 1, 2)
            vout[t0:t1, :, :, :] = block
            del block
            gc.collect()

    ds.close()
    print(f"nc 已保存: {out_path}")


# ============================================================
# 6. baseline 需要的空间插值和光流函数
# ============================================================
def upsample_2x_like_xarray_no_extrap(x):
    """
    x shape: (N, h, w)
    输出 shape: (N, 2h, 2w)

    复现旧 xarray interp 的核心效果：
    - 偶数位置为原值
    - 中间位置线性插值
    - 最后一行/列因为没有外推，保留 NaN
    """

    x = np.asarray(x, dtype=np.float32)
    n, h, w = x.shape

    tmp = np.full((n, 2 * h, w), np.nan, dtype=np.float32)
    tmp[:, 0::2, :] = x
    tmp[:, 1:-1:2, :] = 0.5 * (x[:, :-1, :] + x[:, 1:, :])

    out = np.full((n, 2 * h, 2 * w), np.nan, dtype=np.float32)
    out[:, :, 0::2] = tmp
    out[:, :, 1:-1:2] = 0.5 * (tmp[:, :, :-1] + tmp[:, :, 1:])

    return out


def fill_edge_nan_for_cv2(x):
    """
    光流不能稳定处理 NaN。
    插值造成的最后一行/列 NaN 用邻近边界填充。
    仅用于光流计算，不改变 testy。
    """

    y = np.array(x, dtype=np.float32, copy=True)

    if y.shape[1] >= 2:
        y[:, -1, :] = y[:, -2, :]
    if y.shape[2] >= 2:
        y[:, :, -1] = y[:, :, -2]

    np.nan_to_num(y, copy=False, nan=0.0, posinf=0.0, neginf=0.0)

    return y


def optical_flow_half(frame0, frame1, desc="optical flow"):
    """
    frame0/frame1 shape: (N, H, W)
    返回从 frame0 到 frame1 的半步光流外推结果。
    """

    frame0 = np.asarray(frame0, dtype=np.float32)
    frame1 = np.asarray(frame1, dtype=np.float32)

    assert frame0.shape == frame1.shape

    n, h, w = frame0.shape
    out = np.empty((n, h, w), dtype=np.float32)

    grid_x, grid_y = np.meshgrid(
        np.arange(w, dtype=np.float32),
        np.arange(h, dtype=np.float32)
    )

    for i in tqdm(range(n), desc=desc):
        f0 = np.ascontiguousarray(frame0[i])
        f1 = np.ascontiguousarray(frame1[i])

        flow = cv2.calcOpticalFlowFarneback(
            f0,
            f1,
            None,
            0.5,
            3,
            15,
            3,
            5,
            1.1,
            0
        )

        map_x = grid_x + flow[..., 0] * 0.5
        map_y = grid_y + flow[..., 1] * 0.5

        out[i] = cv2.remap(
            f0,
            map_x.astype(np.float32),
            map_y.astype(np.float32),
            interpolation=cv2.INTER_LINEAR,
            borderMode=cv2.BORDER_REPLICATE
        )

    return out

读取已有 memmap 数据。
testx: (10227, 58, 94, 10)
testy: (10227, 116, 188, 25)

ALL_FEATURE_INDEX: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]
SPATIAL_ONLY_FEATURE_INDEX: [0, 4, 5, 9, 10, 14, 15, 19, 20, 24]
MID_FEATURE_INDEX: [1, 2, 3, 6, 7, 8, 11, 12, 13, 16, 17, 18, 21, 22, 23]


In [2]:

# ============================================================
# 代码框 2：
# TensorFlow 配置 + 三种时间降尺度策略的模型路径
# ============================================================

import os
import gc
from pathlib import Path

# 默认 CPU 推理，避免显存 OOM。
# 如果希望 GPU 推理，将 USE_GPU_FOR_INFERENCE 改为 True。
USE_GPU_FOR_INFERENCE = False

if not USE_GPU_FOR_INFERENCE:
    os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf
from tensorflow.keras.models import load_model
from numpy.lib.format import open_memmap

if USE_GPU_FOR_INFERENCE:
    gpus = tf.config.list_physical_devices("GPU")
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception as e:
            print("GPU memory growth 设置失败:", e)
else:
    try:
        tf.config.set_visible_devices([], "GPU")
    except Exception:
        pass

print("TensorFlow version:", tf.__version__)
print("Visible GPUs:", tf.config.list_physical_devices("GPU"))

# ============================================================
# 1. 模型路径
# ============================================================
# 说明：
# 1) 下面默认使用 E:/Dr_Research/mid 中的模型。
# 2) 如果你的 direct 模型保存在 E:/Dr_Research/model 中，resolve_model_path 会自动尝试。
# 3) 路径可以是完整路径，也可以是模型目录名。
# 4) 如果模型目录名不带 _generator，函数会自动尝试补 _generator。

MODEL_ROOT_CANDIDATES = [
    Path(r"E:/Dr_Research/mid"),
    Path(r"E:/Dr_Research/model"),
]

DIRECT_MODEL = r"Auto_MSG_SE_Densenet_EfficentTemp_GAN_ERA5_data_no_tanh_100km_1day_to_50km_6hour_lr0.01_generator"

SEPARATE_MODELS = {
    "time6":  r"Auto_MSG_SE_Densenet_EfficentTemp_GAN_ERA5_data_no_tanh_100km_1day_to_50km_6hour_time6_lr0.01_generator",
    "time12": r"Auto_MSG_SE_Densenet_EfficentTemp_GAN_ERA5_data_no_tanh_100km_1day_to_50km_6hour_time12_lr0.01_generator",
    "time18": r"Auto_MSG_SE_Densenet_EfficentTemp_GAN_ERA5_data_no_tanh_100km_1day_to_50km_6hour_time18_lr0.01_generator",
}

INDIRECT_STEP1_MODEL = r"Auto_MSG_SE_Densenet_EfficentTemp_GAN_ERA5_data_no_tanh_100km_1day_to_50km_12hour_lr0.01_generator"
INDIRECT_STEP2_MODEL = r"Auto_MSG_SE_Densenet_EfficentTemp_GAN_ERA5_data_no_tanh_50km_12hour_to_50km_6hour_lr0.01_generator"

# 预测 batch size。若 GPU/内存足够可以改大。
PRED_BATCH_SIZE = 1

# 中间结果保存目录
TEMPORAL_TMP_DIR = RESULT_DIR / "_tmp_temporal_strategy_MSG_SED_ET_mid"
TEMPORAL_TMP_DIR.mkdir(parents=True, exist_ok=True)

# 是否已经存在预测结果时跳过预测
SKIP_EXISTING_PRED = True

# 是否保存中间的 step1 预测结果
SAVE_STEP1_PRED = True

CUSTOM_OBJECTS = None


def resolve_model_path(model_name_or_path):
    """
    尝试找到 keras SavedModel 目录。
    支持：
    1. 完整路径；
    2. 只写模型目录名；
    3. 自动尝试补 '_generator'。
    """
    raw = str(model_name_or_path)
    p = Path(raw)

    candidates = []

    # 完整路径或相对路径
    candidates.append(p)

    # 如果没带 _generator，也尝试补
    if not raw.endswith("_generator"):
        candidates.append(Path(raw + "_generator"))

    # 在候选根目录下尝试
    for root in MODEL_ROOT_CANDIDATES:
        candidates.append(root / raw)
        if not raw.endswith("_generator"):
            candidates.append(root / (raw + "_generator"))

    # 去重但保留顺序
    unique_candidates = []
    seen = set()
    for c in candidates:
        cs = str(c)
        if cs not in seen:
            seen.add(cs)
            unique_candidates.append(c)

    for c in unique_candidates:
        if c.exists():
            return c

    msg = "没有找到模型路径。尝试过以下路径：\n" + "\n".join([str(c) for c in unique_candidates])
    raise FileNotFoundError(msg)


print("\n========== 模型路径检查 ==========")
print("Direct:", resolve_model_path(DIRECT_MODEL))
for k, v in SEPARATE_MODELS.items():
    print(f"Separate {k}:", resolve_model_path(v))
print("Indirect step1:", resolve_model_path(INDIRECT_STEP1_MODEL))
print("Indirect step2:", resolve_model_path(INDIRECT_STEP2_MODEL))


# ============================================================
# 2. mid 时次通道定义
# ============================================================
# testy 原始 25 通道：
# var0: hour0,6,12,18,24
# var1: hour0,6,12,18,24
# ...
MID_HOURS = [6, 12, 18]

MID_CHANNELS_FULL = []
MID_VAR_BY_FEATURE = []
MID_HOUR_BY_FEATURE = []

for vi, var_name in enumerate(VAR_NAMES):
    for hi, hour in zip([1, 2, 3], MID_HOURS):
        MID_CHANNELS_FULL.append(vi * 5 + hi)
        MID_VAR_BY_FEATURE.append(var_name)
        MID_HOUR_BY_FEATURE.append(hour)

MID_CHANNELS_FULL = np.array(MID_CHANNELS_FULL, dtype=np.int32)
MID_VAR_BY_FEATURE = np.array(MID_VAR_BY_FEATURE, dtype=object)
MID_HOUR_BY_FEATURE = np.array(MID_HOUR_BY_FEATURE, dtype=np.int32)

# 对 15 通道输出模型，其通道顺序为：
# var0: start, middle, end
# var1: start, middle, end
# ...
MID_CHANNELS_15_OUTPUT = np.array([vi * 3 + 1 for vi in range(len(VAR_NAMES))], dtype=np.int32)

# final mid 输出 15 通道：
# var0: time6, time12, time18
# var1: time6, time12, time18
# ...
print("MID_CHANNELS_FULL:", MID_CHANNELS_FULL)
print("MID_VAR_BY_FEATURE:", MID_VAR_BY_FEATURE)
print("MID_HOUR_BY_FEATURE:", MID_HOUR_BY_FEATURE)
print("MID_CHANNELS_15_OUTPUT:", MID_CHANNELS_15_OUTPUT)


# ============================================================
# 3. 预测通用函数
# ============================================================
def load_generator(model_name_or_path):
    model_path = resolve_model_path(model_name_or_path)
    print("读取 generator:", model_path)
    model = load_model(str(model_path), compile=False, custom_objects=CUSTOM_OBJECTS)
    return model, model_path


def check_model_input_shape(model, actual_input_shape, model_label):
    input_shape = model.input_shape
    if isinstance(input_shape, list):
        input_shape = input_shape[0]

    expected_input = tuple(input_shape[1:])
    actual_input = tuple(actual_input_shape)

    if expected_input != actual_input:
        raise ValueError(
            f"{model_label} 模型输入尺寸不匹配。\n"
            f"模型期望: {expected_input}\n"
            f"当前输入: {actual_input}"
        )


def predict_model_to_mid_memmap(model, x, out_path, out_shape, mode, batch_size=1):
    """
    将模型预测结果保存为 mid 15 通道 memmap。

    mode:
    - 'direct': 模型输出 25 通道，抽取 MID_CHANNELS_FULL；
    - 'separate_time6' / 'separate_time12' / 'separate_time18':
        模型输出 15 通道，只取每个变量的 middle 通道，写入 final mid 的对应时次；
    """
    out_path = Path(out_path)

    if out_path.exists() and SKIP_EXISTING_PRED:
        print(f"预测结果已存在，跳过: {out_path}")
        return np.load(out_path, mmap_mode="r+")

    if out_path.exists():
        out_path.unlink()

    pred_mm = open_memmap(out_path, mode="w+", dtype=np.float32, shape=out_shape)

    n = x.shape[0]

    for b0 in tqdm(range(0, n, batch_size), desc=f"Predict {mode}"):
        b1 = min(n, b0 + batch_size)
        batch_x = np.asarray(x[b0:b1], dtype=np.float32)

        batch_pred = model.predict(batch_x, batch_size=batch_size, verbose=0).astype(np.float32)

        if mode == "direct":
            # direct 输出是 25 通道，抽取 time6/time12/time18
            if batch_pred.shape[-1] != 25:
                raise ValueError(f"direct 模型输出通道不是 25: {batch_pred.shape}")
            pred_mm[b0:b1] = batch_pred[..., MID_CHANNELS_FULL]

        elif mode.startswith("separate_"):
            # separate 输出是 15 通道，只抽取每个变量的 middle 通道
            if batch_pred.shape[-1] != 15:
                raise ValueError(f"{mode} 模型输出通道不是 15: {batch_pred.shape}")

            if mode == "separate_time6":
                tpos = 0
            elif mode == "separate_time12":
                tpos = 1
            elif mode == "separate_time18":
                tpos = 2
            else:
                raise ValueError(f"未知 separate mode: {mode}")

            for vi in range(len(VAR_NAMES)):
                final_ch = vi * 3 + tpos
                model_ch = vi * 3 + 1
                pred_mm[b0:b1, :, :, final_ch] = batch_pred[:, :, :, model_ch]

        else:
            raise ValueError(f"未知 mode: {mode}")

        del batch_x, batch_pred
        gc.collect()

    pred_mm.flush()
    return pred_mm


TensorFlow version: 2.10.0
Visible GPUs: []

========== 模型路径检查 ==========
Direct: E:\Dr_Research\model\Auto_MSG_SE_Densenet_EfficentTemp_GAN_ERA5_data_no_tanh_100km_1day_to_50km_6hour_lr0.01_generator
Separate time6: E:\Dr_Research\model\Auto_MSG_SE_Densenet_EfficentTemp_GAN_ERA5_data_no_tanh_100km_1day_to_50km_6hour_time6_lr0.01_generator
Separate time12: E:\Dr_Research\model\Auto_MSG_SE_Densenet_EfficentTemp_GAN_ERA5_data_no_tanh_100km_1day_to_50km_6hour_time12_lr0.01_generator
Separate time18: E:\Dr_Research\model\Auto_MSG_SE_Densenet_EfficentTemp_GAN_ERA5_data_no_tanh_100km_1day_to_50km_6hour_time18_lr0.01_generator
Indirect step1: E:\Dr_Research\model\Auto_MSG_SE_Densenet_EfficentTemp_GAN_ERA5_data_no_tanh_100km_1day_to_50km_12hour_lr0.01_generator
Indirect step2: E:\Dr_Research\model\Auto_MSG_SE_Densenet_EfficentTemp_GAN_ERA5_data_no_tanh_50km_12hour_to_50km_6hour_lr0.01_generator
MID_CHANNELS_FULL: [ 1  2  3  6  7  8 11 12 13 16 17 18 21 22 23]
MID_VAR_BY_FEATURE: ['slp' 'slp' '

In [3]:

# ============================================================
# 代码框 3：
# 构造 testy_mid memmap，仅保留 time6/time12/time18
# ============================================================

TESTY_MID_NPY = TEMPORAL_TMP_DIR / "testy_mid_standardized.npy"
HR_RANGE_MID_NPY = TEMPORAL_TMP_DIR / "hr_range_mid.npy"

n_test, h_hr, w_hr, _ = testy.shape
mid_shape = (n_test, h_hr, w_hr, len(MID_CHANNELS_FULL))

print("testy full shape:", testy.shape)
print("testy mid shape :", mid_shape)

if TESTY_MID_NPY.exists() and SKIP_EXISTING_PRED:
    print("读取已有 testy_mid:", TESTY_MID_NPY)
    testy_mid = np.load(TESTY_MID_NPY, mmap_mode="r")
else:
    if TESTY_MID_NPY.exists():
        TESTY_MID_NPY.unlink()

    testy_mid_mm = open_memmap(TESTY_MID_NPY, mode="w+", dtype=np.float32, shape=mid_shape)

    for k, full_ch in enumerate(MID_CHANNELS_FULL):
        print(f"写入 testy_mid channel {k} <- testy full channel {full_ch}, "
              f"{MID_VAR_BY_FEATURE[k]} time{MID_HOUR_BY_FEATURE[k]}")
        testy_mid_mm[..., k] = np.asarray(testy[..., full_ch], dtype=np.float32)

    testy_mid_mm.flush()
    del testy_mid_mm
    gc.collect()

    testy_mid = np.load(TESTY_MID_NPY, mmap_mode="r")

hr_range_mid = np.asarray(metadata["hr_range_crop"])[MID_CHANNELS_FULL].astype(np.float32)
np.save(HR_RANGE_MID_NPY, hr_range_mid)

print("testy_mid:", testy_mid.shape)
print("hr_range_mid:", hr_range_mid)


testy full shape: (10227, 116, 188, 25)
testy mid shape : (10227, 116, 188, 15)
读取已有 testy_mid: E:\Dr_Research\result\_tmp_temporal_strategy_MSG_SED_ET_mid\testy_mid_standardized.npy
testy_mid: (10227, 116, 188, 15)
hr_range_mid: [38.301132 38.301147 38.301025 21.041756 21.041435 21.040955 29.014025
 29.014095 29.013926 31.27427  31.27423  31.274244 30.637234 30.63699
 30.636963]


In [4]:

# ============================================================
# 代码框 4：
# Direct 直接法预测
# ============================================================
# Direct 模型：100km_1day -> 50km_6hour
# 输入 testx:  (N, 58, 94, 10)
# 输出 full:   (N, 116, 188, 25)
# 保存 mid:   (N, 116, 188, 15)

DIRECT_PRED_MID_NPY = TEMPORAL_TMP_DIR / "pred_mid_Direct_MSG_SED_ET.npy"

direct_model, direct_path = load_generator(DIRECT_MODEL)
check_model_input_shape(direct_model, testx.shape[1:], "Direct")

pred_direct_mid = predict_model_to_mid_memmap(
    model=direct_model,
    x=testx,
    out_path=DIRECT_PRED_MID_NPY,
    out_shape=mid_shape,
    mode="direct",
    batch_size=PRED_BATCH_SIZE
)

print("Direct mid prediction:", pred_direct_mid.shape)

del direct_model
tf.keras.backend.clear_session()
gc.collect()


读取 generator: E:\Dr_Research\model\Auto_MSG_SE_Densenet_EfficentTemp_GAN_ERA5_data_no_tanh_100km_1day_to_50km_6hour_lr0.01_generator
预测结果已存在，跳过: E:\Dr_Research\result\_tmp_temporal_strategy_MSG_SED_ET_mid\pred_mid_Direct_MSG_SED_ET.npy
Direct mid prediction: (10227, 116, 188, 15)


48641

In [5]:

# ============================================================
# 代码框 5：
# Separate 分别法预测
# ============================================================
# 三个模型分别预测 time6/time12/time18。
# 每个模型输出 15 通道：每个变量的 start/middle/end。
# 这里只抽取每个变量的 middle 通道，写入最终 mid 的对应时次。

SEPARATE_PRED_MID_NPY = TEMPORAL_TMP_DIR / "pred_mid_Separate_MSG_SED_ET.npy"

if SEPARATE_PRED_MID_NPY.exists() and SKIP_EXISTING_PRED:
    print("预测结果已存在，跳过 Separate:", SEPARATE_PRED_MID_NPY)
    pred_separate_mid = np.load(SEPARATE_PRED_MID_NPY, mmap_mode="r+")
else:
    if SEPARATE_PRED_MID_NPY.exists():
        SEPARATE_PRED_MID_NPY.unlink()

    pred_separate_mid = open_memmap(
        SEPARATE_PRED_MID_NPY,
        mode="w+",
        dtype=np.float32,
        shape=mid_shape
    )

    for time_key in ["time6", "time12", "time18"]:
        print(f"\n========== Separate {time_key} ==========")

        sep_model, sep_path = load_generator(SEPARATE_MODELS[time_key])
        check_model_input_shape(sep_model, testx.shape[1:], f"Separate {time_key}")

        # 临时预测到一个同 shape memmap，但只会填一个时次位置
        # 为减少临时文件，直接逐 batch 写入 pred_separate_mid。
        n = testx.shape[0]

        if time_key == "time6":
            tpos = 0
        elif time_key == "time12":
            tpos = 1
        elif time_key == "time18":
            tpos = 2
        else:
            raise ValueError(time_key)

        for b0 in tqdm(range(0, n, PRED_BATCH_SIZE), desc=f"Predict Separate {time_key}"):
            b1 = min(n, b0 + PRED_BATCH_SIZE)
            batch_x = np.asarray(testx[b0:b1], dtype=np.float32)

            batch_pred = sep_model.predict(
                batch_x,
                batch_size=PRED_BATCH_SIZE,
                verbose=0
            ).astype(np.float32)

            if batch_pred.shape[-1] != 15:
                raise ValueError(f"Separate {time_key} 输出通道不是 15: {batch_pred.shape}")

            for vi in range(len(VAR_NAMES)):
                final_ch = vi * 3 + tpos
                model_ch = vi * 3 + 1
                pred_separate_mid[b0:b1, :, :, final_ch] = batch_pred[:, :, :, model_ch]

            del batch_x, batch_pred
            gc.collect()

        pred_separate_mid.flush()

        del sep_model
        tf.keras.backend.clear_session()
        gc.collect()

print("Separate mid prediction:", pred_separate_mid.shape)


预测结果已存在，跳过 Separate: E:\Dr_Research\result\_tmp_temporal_strategy_MSG_SED_ET_mid\pred_mid_Separate_MSG_SED_ET.npy
Separate mid prediction: (10227, 116, 188, 15)


In [6]:

# ============================================================
# 代码框 6：
# Indirect 间接法预测
# ============================================================
# Step 1:
#   100km_1day -> 50km_12hour
#   输出 15 通道：每个变量 [t0, t12, t24]
#
# Step 2:
#   50km_12hour -> 50km_6hour
#   对 [t0, t12] 预测 t6；
#   对 [t12, t24] 预测 t18；
#   t12 默认采用 Step 1 的 t12 结果。
#
# 最终保存 15 通道：
#   每个变量 [time6, time12, time18]
# ============================================================

STEP1_PRED_NPY = TEMPORAL_TMP_DIR / "pred_step1_100km1day_to_50km12hour.npy"
INDIRECT_PRED_MID_NPY = TEMPORAL_TMP_DIR / "pred_mid_Indirect_MSG_SED_ET.npy"

# ------------------------------------------------------------
# 1. Step 1 预测
# ------------------------------------------------------------
if STEP1_PRED_NPY.exists() and SKIP_EXISTING_PRED:
    print("Step1 预测结果已存在，跳过:", STEP1_PRED_NPY)
    pred_step1 = np.load(STEP1_PRED_NPY, mmap_mode="r")
else:
    if STEP1_PRED_NPY.exists():
        STEP1_PRED_NPY.unlink()

    pred_step1 = open_memmap(
        STEP1_PRED_NPY,
        mode="w+",
        dtype=np.float32,
        shape=(n_test, h_hr, w_hr, 15)
    )

    step1_model, step1_path = load_generator(INDIRECT_STEP1_MODEL)
    check_model_input_shape(step1_model, testx.shape[1:], "Indirect Step1")

    for b0 in tqdm(range(0, n_test, PRED_BATCH_SIZE), desc="Predict Indirect Step1"):
        b1 = min(n_test, b0 + PRED_BATCH_SIZE)
        batch_x = np.asarray(testx[b0:b1], dtype=np.float32)

        batch_pred = step1_model.predict(
            batch_x,
            batch_size=PRED_BATCH_SIZE,
            verbose=0
        ).astype(np.float32)

        if batch_pred.shape[1:] != (h_hr, w_hr, 15):
            raise ValueError(
                f"Step1 输出维度错误: {batch_pred.shape}, "
                f"期望: {(b1-b0, h_hr, w_hr, 15)}"
            )

        pred_step1[b0:b1] = batch_pred

        del batch_x, batch_pred
        gc.collect()

    pred_step1.flush()

    del step1_model
    tf.keras.backend.clear_session()
    gc.collect()

# ------------------------------------------------------------
# 2. Step 2 输入构造函数
# ------------------------------------------------------------
def make_step2_input_from_step1(batch_step1, interval):
    """
    batch_step1: (B, H, W, 15)
    通道顺序：每个变量 [t0, t12, t24]

    interval:
    - "0_12": 输入 [t0, t12]，用于预测 time6；
    - "12_24": 输入 [t12, t24]，用于预测 time18。

    返回:
    step2_x: (B, H, W, 10)
    通道顺序：每个变量 [start, end]
    """
    batch_step1 = np.asarray(batch_step1, dtype=np.float32)
    B, H, W, C = batch_step1.shape
    if C != 15:
        raise ValueError(f"batch_step1 通道数应为 15，实际为 {C}")

    step2_x = np.zeros((B, H, W, 10), dtype=np.float32)

    for vi in range(len(VAR_NAMES)):
        if interval == "0_12":
            ch_start = vi * 3 + 0
            ch_end = vi * 3 + 1
        elif interval == "12_24":
            ch_start = vi * 3 + 1
            ch_end = vi * 3 + 2
        else:
            raise ValueError(interval)

        step2_x[..., vi * 2 + 0] = batch_step1[..., ch_start]
        step2_x[..., vi * 2 + 1] = batch_step1[..., ch_end]

    return step2_x


# ------------------------------------------------------------
# 3. Step 2 预测，组装最终 mid 结果
# ------------------------------------------------------------
if INDIRECT_PRED_MID_NPY.exists() and SKIP_EXISTING_PRED:
    print("Indirect mid 预测结果已存在，跳过:", INDIRECT_PRED_MID_NPY)
    pred_indirect_mid = np.load(INDIRECT_PRED_MID_NPY, mmap_mode="r+")
else:
    if INDIRECT_PRED_MID_NPY.exists():
        INDIRECT_PRED_MID_NPY.unlink()

    pred_indirect_mid = open_memmap(
        INDIRECT_PRED_MID_NPY,
        mode="w+",
        dtype=np.float32,
        shape=mid_shape
    )

    step2_model, step2_path = load_generator(INDIRECT_STEP2_MODEL)

    # 这里 Step2 输入是 50km/12hour 的高分辨率端点，shape 应为 (116,188,10)
    step2_expected_input = (h_hr, w_hr, 10)
    check_model_input_shape(step2_model, step2_expected_input, "Indirect Step2")

    for b0 in tqdm(range(0, n_test, PRED_BATCH_SIZE), desc="Predict Indirect Step2"):
        b1 = min(n_test, b0 + PRED_BATCH_SIZE)

        batch_step1 = np.asarray(pred_step1[b0:b1], dtype=np.float32)

        # t12：直接采用 Step1 的中间 12h 结果
        for vi in range(len(VAR_NAMES)):
            final_ch_time12 = vi * 3 + 1
            step1_ch_time12 = vi * 3 + 1
            pred_indirect_mid[b0:b1, :, :, final_ch_time12] = batch_step1[:, :, :, step1_ch_time12]

        # time6: interval [t0, t12]
        x_0_12 = make_step2_input_from_step1(batch_step1, interval="0_12")
        pred_0_12 = step2_model.predict(
            x_0_12,
            batch_size=PRED_BATCH_SIZE,
            verbose=0
        ).astype(np.float32)

        # time18: interval [t12, t24]
        x_12_24 = make_step2_input_from_step1(batch_step1, interval="12_24")
        pred_12_24 = step2_model.predict(
            x_12_24,
            batch_size=PRED_BATCH_SIZE,
            verbose=0
        ).astype(np.float32)

        if pred_0_12.shape[-1] != 15 or pred_12_24.shape[-1] != 15:
            raise ValueError(f"Step2 输出通道不是 15: {pred_0_12.shape}, {pred_12_24.shape}")

        for vi in range(len(VAR_NAMES)):
            # Step2 输出每个变量 [start, middle, end]，middle 是 6h
            model_mid_ch = vi * 3 + 1

            final_ch_time6 = vi * 3 + 0
            final_ch_time18 = vi * 3 + 2

            pred_indirect_mid[b0:b1, :, :, final_ch_time6] = pred_0_12[:, :, :, model_mid_ch]
            pred_indirect_mid[b0:b1, :, :, final_ch_time18] = pred_12_24[:, :, :, model_mid_ch]

        del batch_step1, x_0_12, x_12_24, pred_0_12, pred_12_24
        gc.collect()

    pred_indirect_mid.flush()

    del step2_model
    tf.keras.backend.clear_session()
    gc.collect()

print("Indirect mid prediction:", pred_indirect_mid.shape)


Step1 预测结果已存在，跳过: E:\Dr_Research\result\_tmp_temporal_strategy_MSG_SED_ET_mid\pred_step1_100km1day_to_50km12hour.npy
Indirect mid 预测结果已存在，跳过: E:\Dr_Research\result\_tmp_temporal_strategy_MSG_SED_ET_mid\pred_mid_Indirect_MSG_SED_ET.npy
Indirect mid prediction: (10227, 116, 188, 15)


In [7]:
# ============================================================
# 代码框 7：
# 计算 Direct / Separate / Indirect 的 mid 指标并保存
#
# 说明：
# 1. R、MSE、SSIM、PSNR 均排除最外侧一圈网格。
# 2. SSIM、PSNR 的计算方法与 baseline 对比 notebook 完全一致。
# 3. 只需要重新运行本代码框，无需重新预测。
# ============================================================

def calc_mid_feature_r_mse_spatial_chunk(
    y_true_mid,
    y_pred_mid,
    feature_index,
    lat_chunk=12
):
    """
    y_true_mid/y_pred_mid shape:
        (N, H, W, 15)

    对单个 mid feature 计算：
    1. 每个格点沿 time 维计算 Pearson R，然后对空间格点平均；
    2. 每个格点沿 time 维计算 MSE，然后对空间格点平均；
    3. 排除空间最外侧一圈格点，即 [1:-1, 1:-1]。
    """
    n, h, w, c = y_true_mid.shape

    r_sum = 0.0
    r_count = 0

    mse_sum = 0.0
    mse_count = 0

    # 纬度方向分块，避免一次读取整个 memmap 造成内存峰值
    # i0 从 1 开始，i1 最大到 h-1，从而排除最外侧纬度格点
    for i0 in range(1, h - 1, lat_chunk):
        i1 = min(h - 1, i0 + lat_chunk)

        # 经度方向同样排除最外侧一圈格点
        yt = np.asarray(
            y_true_mid[:, i0:i1, 1:-1, feature_index],
            dtype=np.float64
        )

        yp = np.asarray(
            y_pred_mid[:, i0:i1, 1:-1, feature_index],
            dtype=np.float64
        )

        # ----------------------------------------------------
        # MSE：
        # 先在时间维上计算每个格点的 MSE，再对所有有效格点平均
        # ----------------------------------------------------
        mse_map = np.nanmean((yp - yt) ** 2, axis=0)

        valid_mse = np.isfinite(mse_map)

        if np.any(valid_mse):
            mse_sum += float(
                np.nansum(mse_map[valid_mse])
            )
            mse_count += int(
                np.sum(valid_mse)
            )

        # ----------------------------------------------------
        # Pearson R：
        # 每个格点沿时间维计算相关系数，再对所有有效格点平均
        # ----------------------------------------------------
        yt_mean = np.nanmean(
            yt,
            axis=0,
            keepdims=True
        )

        yp_mean = np.nanmean(
            yp,
            axis=0,
            keepdims=True
        )

        yt_anom = yt - yt_mean
        yp_anom = yp - yp_mean

        numerator = np.nansum(
            yt_anom * yp_anom,
            axis=0
        )

        denominator = np.sqrt(
            np.nansum(yt_anom ** 2, axis=0)
            *
            np.nansum(yp_anom ** 2, axis=0)
        )

        # 某些时间序列方差为 0 时，相关系数会产生 NaN/Inf；
        # 后续仅统计有限值
        with np.errstate(
            divide="ignore",
            invalid="ignore"
        ):
            r_map = numerator / denominator

        valid_r = np.isfinite(r_map)

        if np.any(valid_r):
            r_sum += float(
                np.nansum(r_map[valid_r])
            )
            r_count += int(
                np.sum(valid_r)
            )

        del (
            yt,
            yp,
            yt_mean,
            yp_mean,
            yt_anom,
            yp_anom,
            numerator,
            denominator,
            r_map,
            mse_map,
            valid_r,
            valid_mse
        )
        gc.collect()

    r = (
        r_sum / r_count
        if r_count > 0
        else np.nan
    )

    mse = (
        mse_sum / mse_count
        if mse_count > 0
        else np.nan
    )

    return float(r), float(mse)


def calc_mid_feature_ssim_psnr_tf(
    y_true_mid,
    y_pred_mid,
    feature_index,
    max_val,
    batch_size=16
):
    """
    使用与 baseline 对比 notebook 完全一致的方法，
    分 batch 计算单个 mid feature 的 SSIM 和 PSNR。

    评价规则：
    1. 排除空间最外侧一圈格点，即 [1:-1, 1:-1]；
    2. 将 NaN、正无穷和负无穷统一替换为 0；
    3. 对每个测试样本分别计算 SSIM 和 PSNR；
    4. 最后对全部测试样本求平均；
    5. 在 CPU 上计算，避免占用训练/推理 GPU。
    """
    import tensorflow as tf

    n = y_true_mid.shape[0]

    if (not np.isfinite(max_val)) or max_val <= 0:
        max_val = 1.0

    ssim_sum = 0.0
    psnr_sum = 0.0
    count = 0

    with tf.device("/CPU:0"):
        for b0 in range(0, n, batch_size):
            b1 = min(n, b0 + batch_size)

            # =================================================
            # 关键修改：
            # 与 baseline 一致，排除空间最外侧一圈格点
            # =================================================
            yt = np.asarray(
                y_true_mid[
                    b0:b1,
                    1:-1,
                    1:-1,
                    feature_index
                ],
                dtype=np.float32
            )

            yp = np.asarray(
                y_pred_mid[
                    b0:b1,
                    1:-1,
                    1:-1,
                    feature_index
                ],
                dtype=np.float32
            )

            # =================================================
            # 与 baseline 一致：
            # NaN、+Inf、-Inf 均替换为 0
            # 并增加单通道维度：
            # (batch, H, W) -> (batch, H, W, 1)
            # =================================================
            yt = np.nan_to_num(
                yt,
                nan=0.0,
                posinf=0.0,
                neginf=0.0
            )[..., np.newaxis]

            yp = np.nan_to_num(
                yp,
                nan=0.0,
                posinf=0.0,
                neginf=0.0
            )[..., np.newaxis]

            # 每个样本分别计算 SSIM
            ssim_batch = tf.image.ssim(
                yp,
                yt,
                max_val=float(max_val)
            ).numpy()

            # 每个样本分别计算 PSNR
            psnr_batch = tf.image.psnr(
                yp,
                yt,
                max_val=float(max_val)
            ).numpy()

            valid_ssim = np.isfinite(ssim_batch)
            valid_psnr = np.isfinite(psnr_batch)

            if np.any(valid_ssim):
                ssim_sum += float(
                    np.nansum(ssim_batch[valid_ssim])
                )

            if np.any(valid_psnr):
                psnr_sum += float(
                    np.nansum(psnr_batch[valid_psnr])
                )

            # 与 baseline 代码一致：
            # count 按当前 batch 的样本数量累加
            count += int(b1 - b0)

            del (
                yt,
                yp,
                ssim_batch,
                psnr_batch,
                valid_ssim,
                valid_psnr
            )
            gc.collect()

    ssim = (
        ssim_sum / count
        if count > 0
        else np.nan
    )

    psnr = (
        psnr_sum / count
        if count > 0
        else np.nan
    )

    return float(ssim), float(psnr)


def calc_mid_feature_metrics_table(
    y_true_mid,
    y_pred_mid,
    model_name,
    ssim_batch_size=16
):
    """
    分别计算 15 个 mid feature 的指标。

    15 个 feature =
        5 个变量 × 3 个中间时次：
        time6、time12、time18。
    """
    rows = []

    if y_true_mid.shape != y_pred_mid.shape:
        raise ValueError(
            f"shape 不一致: "
            f"y_true={y_true_mid.shape}, "
            f"y_pred={y_pred_mid.shape}"
        )

    for k in tqdm(
        range(y_true_mid.shape[-1]),
        desc=f"Mid feature metrics: {model_name}"
    ):
        var_name = str(
            MID_VAR_BY_FEATURE[k]
        )

        hour = int(
            MID_HOUR_BY_FEATURE[k]
        )

        # R 和 MSE
        r, mse = calc_mid_feature_r_mse_spatial_chunk(
            y_true_mid=y_true_mid,
            y_pred_mid=y_pred_mid,
            feature_index=k,
            lat_chunk=12
        )

        # SSIM 和 PSNR
        ssim, psnr = calc_mid_feature_ssim_psnr_tf(
            y_true_mid=y_true_mid,
            y_pred_mid=y_pred_mid,
            feature_index=k,
            max_val=float(hr_range_mid[k]),
            batch_size=ssim_batch_size
        )

        rows.append({
            "Model": model_name,
            "Variable": var_name,
            "Hour": hour,
            "Feature_Index_Mid": k,
            "R": r,
            "MSE": mse,
            "SSIM": ssim,
            "PSNR": psnr,
        })

        gc.collect()

    return pd.DataFrame(rows)


def save_mid_strategy_metrics(
    feature_tables,
    out_xlsx
):
    """
    保存三层指标结果：

    1. mid_overall：
       对 5 个变量和 3 个中间时次整体平均；

    2. mid_by_variable：
       每个变量对 time6、time12、time18 平均；

    3. mid_by_feature：
       每个变量、每个中间时次分别输出。
    """
    metric_cols = [
        "R",
        "MSE",
        "SSIM",
        "PSNR"
    ]

    by_feature = pd.concat(
        list(feature_tables.values()),
        axis=0,
        ignore_index=True
    )

    # 每个变量：
    # 对 time6、time12、time18 的指标平均
    by_variable = (
        by_feature
        .groupby(
            ["Model", "Variable"],
            as_index=False
        )[metric_cols]
        .mean()
    )

    # 总体：
    # 对 5 个变量进一步平均
    overall = (
        by_variable
        .groupby(
            "Model",
            as_index=False
        )[metric_cols]
        .mean()
    )

    # --------------------------------------------------------
    # 固定三种方法顺序
    # --------------------------------------------------------
    order = [
        "Direct",
        "Separate",
        "Indirect"
    ]

    method_order_map = {
        method: i
        for i, method in enumerate(order)
    }

    var_order_map = {
        var: i
        for i, var in enumerate(VAR_NAMES)
    }

    overall["__order"] = overall["Model"].map(
        method_order_map
    )

    overall = (
        overall
        .sort_values("__order")
        .drop(columns="__order")
    )

    by_variable["__order"] = by_variable["Model"].map(
        method_order_map
    )

    by_variable["__var_order"] = by_variable["Variable"].map(
        var_order_map
    )

    by_variable = (
        by_variable
        .sort_values([
            "__order",
            "__var_order"
        ])
        .drop(columns=[
            "__order",
            "__var_order"
        ])
    )

    by_feature["__order"] = by_feature["Model"].map(
        method_order_map
    )

    by_feature["__var_order"] = by_feature["Variable"].map(
        var_order_map
    )

    by_feature = (
        by_feature
        .sort_values([
            "__order",
            "__var_order",
            "Hour"
        ])
        .drop(columns=[
            "__order",
            "__var_order"
        ])
    )

    # --------------------------------------------------------
    # 保存 Excel 和 CSV
    # --------------------------------------------------------
    out_xlsx = Path(out_xlsx)
    out_xlsx.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with pd.ExcelWriter(
        out_xlsx,
        engine="openpyxl"
    ) as writer:
        overall.to_excel(
            writer,
            sheet_name="mid_overall",
            index=False
        )

        by_variable.to_excel(
            writer,
            sheet_name="mid_by_variable",
            index=False
        )

        by_feature.to_excel(
            writer,
            sheet_name="mid_by_feature",
            index=False
        )

    prefix = out_xlsx.with_suffix("")

    overall.to_csv(
        str(prefix) + "_mid_overall.csv",
        index=False,
        encoding="utf-8-sig"
    )

    by_variable.to_csv(
        str(prefix) + "_mid_by_variable.csv",
        index=False,
        encoding="utf-8-sig"
    )

    by_feature.to_csv(
        str(prefix) + "_mid_by_feature.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print(f"\n指标表已保存: {out_xlsx}")

    print("\n========== mid overall ==========")
    try:
        display(overall)
    except Exception:
        print(overall)

    print("\n========== mid by variable ==========")
    try:
        display(by_variable)
    except Exception:
        print(by_variable)

    print("\n========== mid by feature ==========")
    try:
        display(by_feature)
    except Exception:
        print(by_feature)

    return (
        overall,
        by_variable,
        by_feature
    )


# ============================================================
# 读取三种方法已经保存的 mid 预测结果
# ============================================================
pred_paths = {
    "Direct": DIRECT_PRED_MID_NPY,
    "Separate": SEPARATE_PRED_MID_NPY,
    "Indirect": INDIRECT_PRED_MID_NPY,
}

feature_tables = {}

for method_name, pred_path in pred_paths.items():
    print(
        f"\n========== 计算 {method_name} 指标 =========="
    )

    if not Path(pred_path).exists():
        raise FileNotFoundError(
            f"预测文件不存在：{pred_path}\n"
            f"请先运行对应的预测代码框。"
        )

    y_pred_mid = np.load(
        pred_path,
        mmap_mode="r"
    )

    print("y_pred_mid:", y_pred_mid.shape)
    print("testy_mid :", testy_mid.shape)

    table = calc_mid_feature_metrics_table(
        y_true_mid=testy_mid,
        y_pred_mid=y_pred_mid,
        model_name=method_name,
        ssim_batch_size=16
    )

    feature_tables[method_name] = table

    feature_csv_path = (
        RESULT_DIR
        / f"temporal_strategy_{method_name}_MSG_SED_ET_mid_feature_metrics.csv"
    )

    table.to_csv(
        feature_csv_path,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        "feature 级指标已保存:",
        feature_csv_path
    )

    del y_pred_mid
    gc.collect()


# ============================================================
# 汇总并保存最终结果
# ============================================================
out_xlsx = (
    RESULT_DIR
    / "temporal_strategy_MSG_SED_ET_mid_metrics.xlsx"
)

mid_overall, mid_by_variable, mid_by_feature = (
    save_mid_strategy_metrics(
        feature_tables=feature_tables,
        out_xlsx=out_xlsx
    )
)


========== 计算 Direct 指标 ==========
y_pred_mid: (10227, 116, 188, 15)
testy_mid : (10227, 116, 188, 15)


Mid feature metrics: Direct: 100%|████████████████████████████████████████████████████| 15/15 [55:39<00:00, 222.65s/it]


feature 级指标已保存: E:\Dr_Research\result\temporal_strategy_Direct_MSG_SED_ET_mid_feature_metrics.csv

========== 计算 Separate 指标 ==========
y_pred_mid: (10227, 116, 188, 15)
testy_mid : (10227, 116, 188, 15)


Mid feature metrics: Separate: 100%|██████████████████████████████████████████████████| 15/15 [46:03<00:00, 184.25s/it]


feature 级指标已保存: E:\Dr_Research\result\temporal_strategy_Separate_MSG_SED_ET_mid_feature_metrics.csv

========== 计算 Indirect 指标 ==========
y_pred_mid: (10227, 116, 188, 15)
testy_mid : (10227, 116, 188, 15)


...

,Model,R,MSE,SSIM,PSNR
0,Direct,0.871169,0.234941,0.770052,36.420937
2,Separate,0.827743,0.413733,0.687320,34.428767
1,Indirect,0.832175,0.376124,0.694331,34.582571



========== mid by variable ==========


,Model,Variable,R,MSE,SSIM,PSNR
0,Direct,slp,0.887236,0.220554,0.786024,38.759331
3,Direct,z300,0.934823,0.135769,0.820670,35.861965
4,Direct,z500,0.911468,0.170075,0.815122,37.780322
1,Direct,u10,0.816422,0.311599,0.728950,35.127894
2,Direct,v10,0.805897,0.336710,0.699494,34.575173
10,Separate,slp,0.804505,0.436443,0.682400,36.099534
13,Separate,z300,0.903639,0.256650,0.734794,34.204664
14,Separate,z500,0.839228,0.362125,0.707848,35.608245
11,Separate,u10,0.805434,0.485500,0.688582,33.566939
12,Separate,v10,0.785910,0.527947,0.622977,32.664454



========== mid by feature ==========


,Model,Variable,Hour,Feature_Index_Mid,R,MSE,SSIM,PSNR
0,Direct,slp,6,0,0.872158,0.250802,0.761665,38.344749
1,Direct,slp,12,1,0.914204,0.175678,0.822997,39.562976
2,Direct,slp,18,2,0.875345,0.235182,0.773411,38.370267
3,Direct,z300,6,3,0.920762,0.159432,0.798548,35.068697
4,Direct,z300,12,4,0.963598,0.080380,0.870251,37.730098
5,Direct,z300,18,5,0.920107,0.167494,0.793211,34.787101
6,Direct,z500,6,6,0.886753,0.211441,0.780359,36.636611
7,Direct,z500,12,7,0.962145,0.087029,0.885954,40.129915
8,Direct,z500,18,8,0.885506,0.211756,0.779052,36.574439
9,Direct,u10,6,9,0.829664,0.292528,0.743513,35.423312


In [8]:

# ============================================================
# 代码框 8（可选）：
# 保存 testy_mid 和三种方法 pred_mid 为 nc
# ============================================================
# 每个变量 shape: (time, feature, latitude, longitude)
# feature 维度为 [time6, time12, time18]
# ============================================================

SAVE_MID_NC = True

def save_mid_multivar_to_nc(arr, times, lat, lon, out_path, description, time_chunk=64, complevel=4):
    """
    arr shape: (N, H, W, 15)
    每个变量 3 个 feature: time6, time12, time18
    保存为 data_slp/data_z300/.../data_v10，shape=(time, feature, latitude, longitude)
    """
    import xarray as xr

    arr = np.asarray(arr)
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    data_vars = {}

    for vi, var_name in enumerate(VAR_NAMES):
        ch0 = vi * 3
        ch1 = ch0 + 3

        data_vars[f"data_{var_name}"] = (
            ("time", "feature", "latitude", "longitude"),
            np.transpose(arr[:, :, :, ch0:ch1], (0, 3, 1, 2))
        )

    ds = xr.Dataset(
        data_vars=data_vars,
        coords={
            "time": times,
            "feature": np.array(MID_HOURS, dtype=np.int32),
            "latitude": lat,
            "longitude": lon,
        },
        attrs={
            "description": description,
            "note": "feature = 6, 12, 18 hours within each 24-hour downscaling window.",
        }
    )

    encoding = {}
    for v in ds.data_vars:
        encoding[v] = {
            "zlib": True,
            "complevel": complevel,
            "chunksizes": (min(time_chunk, arr.shape[0]), 3, arr.shape[1], arr.shape[2])
        }

    ds.to_netcdf(out_path, encoding=encoding)
    ds.close()
    print("nc 已保存:", out_path)


if SAVE_MID_NC:
    lat_hr = metadata["lat"]
    lon_hr = metadata["lon"]
    test_times = metadata["test_times"]

    save_mid_multivar_to_nc(
        arr=testy_mid,
        times=test_times,
        lat=lat_hr,
        lon=lon_hr,
        out_path=RESULT_DIR / "testy_mid_standardized.nc",
        description="Standardized HR label data for mid time steps only."
    )

    for method_name, pred_path in pred_paths.items():
        y_pred_mid = np.load(pred_path, mmap_mode="r")
        save_mid_multivar_to_nc(
            arr=y_pred_mid,
            times=test_times,
            lat=lat_hr,
            lon=lon_hr,
            out_path=RESULT_DIR / f"pred_mid_{method_name}_MSG_SED_ET_standardized.nc",
            description=f"Standardized mid-time prediction from {method_name} strategy using MSG-SE-DenseNet-EfficentTemp-GAN."
        )
        del y_pred_mid
        gc.collect()

print("\n完成。")


nc 已保存: E:\Dr_Research\result\testy_mid_standardized.nc
nc 已保存: E:\Dr_Research\result\pred_mid_Direct_MSG_SED_ET_standardized.nc
nc 已保存: E:\Dr_Research\result\pred_mid_Separate_MSG_SED_ET_standardized.nc
nc 已保存: E:\Dr_Research\result\pred_mid_Indirect_MSG_SED_ET_standardized.nc

完成。
